In [ ]:
docs = [
    "The company announced a dividend.",
    "The company announced a new contract.",
    "The board approved the dividend payment.",
    "The company signed a contract with a bank.",
    "The company announced results, the company announced a new factory, and the company announced a dividend.",
]

docs.append(docs[0]*10)

query = "dividend payment"

In [19]:
import re
import numpy as np

stop = {"the", "a", "with", "and"}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [w for w in words if w not in stop]

In [20]:
# 1) Her cümleyi kelimelerine ayır
tokens = [tokenize(d) for d in docs]

# 2) Kelime listesi (sütunlar)
vocab = sorted({w for d in tokens for w in d})

idx = {}
for j, w in enumerate(vocab):
    idx[w] = j

# 3) A matrisi: satır = cümle, sütun = kelime, hücre = kaç kez geçtiği
A = np.zeros((len(docs), len(vocab)))
for i, d in enumerate(tokens):
    for w in d:
        A[i, idx[w]] += 1

# 4) Sorgu vektörü — matristeki her satır ile ayni elemana sahip bir vektör içinde sadece aranan kelimelerin pozisyonu 1
q = np.zeros(len(vocab))
for w in tokenize(query):
    if w in idx:
        q[idx[w]] += 1


# Aq sonucu her eleman bir cümlede aranan kelimenin kaç tanesi olduğunu ifade eder.
# Aranan kelimeler dividend ve payment.
print(vocab)
print(A)
print(q)

# 1. cümlede 1 tanesi var, 3. cümlede 2'side var, 5. cümlede 1 tanesi var
print("Aq =", A @ q)

['announced', 'approved', 'bank', 'board', 'company', 'contract', 'dividend', 'factory', 'new', 'payment', 'results', 'signed']
[[ 1.  0.  0.  0.  1.  0.  1.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  1.  1.  0.  0.  1.  0.  0.  0.]
 [ 0.  1.  0.  1.  0.  0.  1.  0.  0.  1.  0.  0.]
 [ 0.  0.  1.  0.  1.  1.  0.  0.  0.  0.  0.  1.]
 [ 3.  0.  0.  0.  3.  0.  1.  1.  1.  0.  1.  0.]
 [30.  0.  0.  0. 30.  0. 10. 10. 10.  0. 10.  0.]]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0.]
Aq = [ 1.  0.  2.  0.  1. 10.]


In [21]:
# find the l2 norms of each sentence in A and q
row_norms = []
for i in range(len(A)):
    squared = 0
    for word in A[i]:
        squared += word**2
    row_norms.append(pow(squared, 0.5))

q_norm = 0
for i in q:
    q_norm += i**2
q_norm = pow(q_norm, 0.5)

print(row_norms)
print(q_norm)

[np.float64(1.7320508075688772), np.float64(2.0), np.float64(2.0), np.float64(2.0), np.float64(4.69041575982343), np.float64(46.9041575982343)]
1.4142135623730951


In [22]:
cos_similarities = []
for i in range(len(A)):
    sim = (A[i] @ q) / (row_norms[i] * q_norm)
    cos_similarities.append(sim)

# A vektöründeki her satırın q deki her satır ile olan cos benzerliği ölçülür
print(cos_similarities)

[np.float64(0.40824829046386296), np.float64(0.0), np.float64(0.7071067811865475), np.float64(0.0), np.float64(0.1507556722888818), np.float64(0.1507556722888818)]
